# Activations: tanh and ReLU

**Goal:** Implement sigmoid, tanh, ReLU, LeakyReLU, and GELU from scratch in PyTorch,
derive their analytic derivatives, validate against `torch` / `torch.autograd`, plot
activation curves and derivatives, and discuss the practical consequences of each
activation's properties for deep-network training.

Topics: saturation (sigmoid/tanh), dead-ReLU problem, why ReLU/GELU dominate in
deep nets, and the link between activation choice and gradient flow.

## Configuration

Device, random seed, and default dtype come from `shared.config.configure()`.
Plots use `matplotlib` with the non-interactive `Agg` backend so the notebook
executes headlessly.

In [1]:
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # noqa
import matplotlib.pyplot as plt  # noqa: E402
import torch  # noqa: E402
import torch.nn.functional as F  # noqa: E402


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

from shared.config import configure  # noqa: E402

device = configure()
print("running on:", device)

running on: mps


## From Scratch: Activation Functions and Their Derivatives

We implement each activation and its analytic derivative using only
`torch` primitives — no `torch.nn.functional` calls in these functions.

### Formulae recap

| Activation | Forward | Derivative |
|---|---|---|
| sigmoid | `σ(x) = 1 / (1 + exp(-x))` | `σ(x) · (1 - σ(x))` |
| tanh | `(exp(x) - exp(-x)) / (exp(x) + exp(-x))` | `1 - tanh(x)²` |
| ReLU | `max(0, x)` | `1 if x > 0, else 0` |
| LeakyReLU | `max(αx, x)`, α = 0.01 | `1 if x > 0, else α` |
| GELU | `x · Φ(x)` (Φ = standard normal CDF) | `Φ(x) + x · φ(x)` |

For GELU we use the exact form via `0.5 * (1 + erf(x / sqrt(2)))`.

In [2]:
import math

LEAKY_ALPHA = 0.01


def sigmoid_scratch(x: torch.Tensor) -> torch.Tensor:
    """Sigmoid: σ(x) = 1 / (1 + exp(-x))."""
    return 1.0 / (1.0 + torch.exp(-x))


def sigmoid_deriv_scratch(x: torch.Tensor) -> torch.Tensor:
    """Analytic derivative of sigmoid: σ(x) * (1 - σ(x))."""
    s = sigmoid_scratch(x)
    return s * (1.0 - s)


def tanh_scratch(x: torch.Tensor) -> torch.Tensor:
    """tanh(x) = (exp(x) - exp(-x)) / (exp(x) + exp(-x))."""
    ep = torch.exp(x)
    em = torch.exp(-x)
    return (ep - em) / (ep + em)


def tanh_deriv_scratch(x: torch.Tensor) -> torch.Tensor:
    """Analytic derivative of tanh: 1 - tanh(x)^2."""
    t = tanh_scratch(x)
    return 1.0 - t ** 2


def relu_scratch(x: torch.Tensor) -> torch.Tensor:
    """ReLU: max(0, x)."""
    return torch.clamp(x, min=0.0)


def relu_deriv_scratch(x: torch.Tensor) -> torch.Tensor:
    """Subgradient of ReLU: 1 where x > 0, 0 elsewhere (0 at x=0 by convention)."""
    return (x > 0).to(x.dtype)


def leaky_relu_scratch(x: torch.Tensor, alpha: float = LEAKY_ALPHA) -> torch.Tensor:
    """LeakyReLU: max(alpha * x, x)."""
    return torch.where(x >= 0, x, alpha * x)


def leaky_relu_deriv_scratch(x: torch.Tensor, alpha: float = LEAKY_ALPHA) -> torch.Tensor:
    """Derivative of LeakyReLU: 1 where x >= 0, alpha elsewhere."""
    return torch.where(x >= 0, torch.ones_like(x), torch.full_like(x, alpha))


_SQRT2 = math.sqrt(2.0)


def gelu_scratch(x: torch.Tensor) -> torch.Tensor:
    """GELU (exact): x * Phi(x) where Phi is the standard normal CDF.

    Phi(x) = 0.5 * (1 + erf(x / sqrt(2))).
    """
    return x * 0.5 * (1.0 + torch.erf(x / _SQRT2))


def gelu_deriv_scratch(x: torch.Tensor) -> torch.Tensor:
    """Analytic derivative of GELU: Phi(x) + x * phi(x).

    phi(x) = exp(-x^2/2) / sqrt(2*pi)  (standard normal PDF).
    """
    phi_cdf = 0.5 * (1.0 + torch.erf(x / _SQRT2))
    phi_pdf = torch.exp(-0.5 * x ** 2) / math.sqrt(2.0 * math.pi)
    return phi_cdf + x * phi_pdf


# Sample inputs spanning saturation + active regions
x = torch.linspace(-4, 4, 400, device=device)

print("sigmoid range:    [{:.4f}, {:.4f}]".format(sigmoid_scratch(x).min().item(), sigmoid_scratch(x).max().item()))
print("tanh range:       [{:.4f}, {:.4f}]".format(tanh_scratch(x).min().item(), tanh_scratch(x).max().item()))
print("relu(x) at x=-2:", relu_scratch(torch.tensor([-2.0], device=device)).item())
print("relu(x) at x=+2:", relu_scratch(torch.tensor([2.0], device=device)).item())
print("gelu(0):", gelu_scratch(torch.tensor([0.0], device=device)).item())

sigmoid range:    [0.0180, 0.9820]
tanh range:       [-0.9993, 0.9993]
relu(x) at x=-2: 0.0
relu(x) at x=+2: 2.0
gelu(0): 0.0


## Validation: Forward Pass vs. torch Built-ins

We assert element-wise agreement between our scratch implementations and
`torch.sigmoid`, `torch.tanh`, `F.relu`, `F.leaky_relu`, and `F.gelu`.

In [3]:
x_val = torch.linspace(-5, 5, 1000, device=device)

atol = 1e-5

# sigmoid
diff_sig = (sigmoid_scratch(x_val) - torch.sigmoid(x_val)).abs().max().item()
assert diff_sig < atol, f"sigmoid mismatch: max diff = {diff_sig}"

# tanh
diff_tanh = (tanh_scratch(x_val) - torch.tanh(x_val)).abs().max().item()
assert diff_tanh < atol, f"tanh mismatch: max diff = {diff_tanh}"

# relu
diff_relu = (relu_scratch(x_val) - F.relu(x_val)).abs().max().item()
assert diff_relu < atol, f"relu mismatch: max diff = {diff_relu}"

# leaky relu
diff_lrelu = (leaky_relu_scratch(x_val) - F.leaky_relu(x_val, negative_slope=LEAKY_ALPHA)).abs().max().item()
assert diff_lrelu < atol, f"leaky_relu mismatch: max diff = {diff_lrelu}"

# gelu (exact)
diff_gelu = (gelu_scratch(x_val) - F.gelu(x_val)).abs().max().item()
assert diff_gelu < atol, f"gelu mismatch: max diff = {diff_gelu}"

print("All forward-pass assertions passed.")
print(f"  sigmoid   max|diff| = {diff_sig:.2e}")
print(f"  tanh      max|diff| = {diff_tanh:.2e}")
print(f"  relu      max|diff| = {diff_relu:.2e}")
print(f"  leaky_relu max|diff| = {diff_lrelu:.2e}")
print(f"  gelu      max|diff| = {diff_gelu:.2e}")

All forward-pass assertions passed.
  sigmoid   max|diff| = 0.00e+00
  tanh      max|diff| = 1.79e-07
  relu      max|diff| = 0.00e+00
  leaky_relu max|diff| = 0.00e+00
  gelu      max|diff| = 9.54e-07


## Validation: Analytic Derivatives vs. `torch.autograd.grad`

For each activation `f`, we compute `df/dx` via our analytic formula and
compare it to the gradient returned by `torch.autograd.grad(f(x).sum(), x)`.

In [4]:
x_ag = torch.linspace(-4, 4, 500, device=device, requires_grad=True)

def autograd_deriv(fn, x):
    """Compute df/dx using autograd."""
    y = fn(x.detach().clone().requires_grad_(True))
    (g,) = torch.autograd.grad(y.sum(), y.grad_fn.next_functions[0][0].variable
                                if hasattr(y.grad_fn, 'next_functions') else y,
                                allow_unused=True)
    # simpler: just use backward on a fresh leaf
    xf = x.detach().clone().requires_grad_(True)
    out = fn(xf)
    out.sum().backward()
    return xf.grad.detach()

def check_deriv(name, fn_scratch, deriv_scratch, fn_torch):
    xf = x_ag.detach().clone().requires_grad_(True)
    out = fn_torch(xf)
    out.sum().backward()
    ag_grad = xf.grad.detach()
    my_grad = deriv_scratch(x_ag.detach())
    diff = (my_grad - ag_grad).abs().max().item()
    assert diff < 1e-4, f"{name} derivative mismatch vs autograd: max diff = {diff}"
    print(f"  {name:12s}  analytic vs autograd  max|diff| = {diff:.2e}")

print("Analytic derivative vs torch.autograd.grad:")
check_deriv("sigmoid",    sigmoid_scratch,     sigmoid_deriv_scratch,     torch.sigmoid)
check_deriv("tanh",       tanh_scratch,        tanh_deriv_scratch,        torch.tanh)
check_deriv("relu",       relu_scratch,        relu_deriv_scratch,        F.relu)
check_deriv("leaky_relu", leaky_relu_scratch,  leaky_relu_deriv_scratch,
            lambda x: F.leaky_relu(x, negative_slope=LEAKY_ALPHA))
check_deriv("gelu",       gelu_scratch,        gelu_deriv_scratch,        F.gelu)

print("All derivative assertions passed.")

Analytic derivative vs torch.autograd.grad:


  sigmoid       analytic vs autograd  max|diff| = 0.00e+00
  tanh          analytic vs autograd  max|diff| = 3.58e-07


  relu          analytic vs autograd  max|diff| = 0.00e+00
  leaky_relu    analytic vs autograd  max|diff| = 0.00e+00
  gelu          analytic vs autograd  max|diff| = 1.19e-07
All derivative assertions passed.


## Idiomatic PyTorch: Using `torch.nn.functional` Directly

In production code, prefer the built-in functions — they are fused, hardware-optimised,
and produce correct autograd graphs automatically.

In [5]:
x_demo = torch.linspace(-3, 3, 7, device=device)

print("x              :", x_demo.tolist())
print("torch.sigmoid  :", torch.sigmoid(x_demo).tolist())
print("torch.tanh     :", torch.tanh(x_demo).tolist())
print("F.relu         :", F.relu(x_demo).tolist())
print("F.leaky_relu   :", F.leaky_relu(x_demo, negative_slope=0.01).tolist())
print("F.gelu         :", F.gelu(x_demo).tolist())

# nn.Module equivalents for use inside model definitions
import torch.nn as nn

activations = [nn.Sigmoid(), nn.Tanh(), nn.ReLU(), nn.LeakyReLU(0.01), nn.GELU()]
for act in activations:
    out = act(x_demo)
    print(f"{act.__class__.__name__:12s}: {out.tolist()}")

x              : [-3.0, -2.0, -1.0, 0.0, 1.0, 2.0, 3.0]
torch.sigmoid  : [0.04742587357759476, 0.11920291930437088, 0.26894140243530273, 0.5, 0.7310585975646973, 0.8807970285415649, 0.9525741338729858]
torch.tanh     : [-0.9950547218322754, -0.9640276432037354, -0.7615941166877747, 0.0, 0.7615941166877747, 0.9640276432037354, 0.9950547218322754]
F.relu         : [0.0, 0.0, 0.0, 0.0, 1.0, 2.0, 3.0]
F.leaky_relu   : [-0.029999999329447746, -0.019999999552965164, -0.009999999776482582, 0.0, 1.0, 2.0, 3.0]
F.gelu         : [-0.0040496885776519775, -0.04550027847290039, -0.15865525603294373, 0.0, 0.8413447141647339, 1.9544997215270996, 2.995950222015381]
Sigmoid     : [0.04742587357759476, 0.11920291930437088, 0.26894140243530273, 0.5, 0.7310585975646973, 0.8807970285415649, 0.9525741338729858]
Tanh        : [-0.9950547218322754, -0.9640276432037354, -0.7615941166877747, 0.0, 0.7615941166877747, 0.9640276432037354, 0.9950547218322754]
ReLU        : [0.0, 0.0, 0.0, 0.0, 1.0, 2.0, 3.0]
LeakyR

## Plots: Activation Curves and Derivatives

We visualise all five activations and their derivatives over `[-4, 4]`.

In [6]:
x_plot = torch.linspace(-4, 4, 400, device=device)
x_np = x_plot.cpu().numpy()

act_fns = {
    "sigmoid":    (sigmoid_scratch, sigmoid_deriv_scratch),
    "tanh":       (tanh_scratch,    tanh_deriv_scratch),
    "relu":       (relu_scratch,    relu_deriv_scratch),
    "leaky_relu": (leaky_relu_scratch, leaky_relu_deriv_scratch),
    "gelu":       (gelu_scratch,    gelu_deriv_scratch),
}

fig, axes = plt.subplots(2, 5, figsize=(20, 7))

for col, (name, (fn, dfn)) in enumerate(act_fns.items()):
    y     = fn(x_plot).cpu().detach().numpy()
    dy    = dfn(x_plot).cpu().detach().numpy()

    ax_top = axes[0, col]
    ax_top.plot(x_np, y, lw=2)
    ax_top.axhline(0, color="k", lw=0.5, ls="--")
    ax_top.axvline(0, color="k", lw=0.5, ls="--")
    ax_top.set_title(name)
    ax_top.set_ylim(-1.5, 1.5)
    if col == 0:
        ax_top.set_ylabel("f(x)")

    ax_bot = axes[1, col]
    ax_bot.plot(x_np, dy, lw=2, color="tab:orange")
    ax_bot.axhline(0, color="k", lw=0.5, ls="--")
    ax_bot.axvline(0, color="k", lw=0.5, ls="--")
    ax_bot.set_ylim(-0.2, 1.2)
    if col == 0:
        ax_bot.set_ylabel("f'(x)")

fig.suptitle("Activation Functions (top) and Their Derivatives (bottom)", fontsize=13)
plt.tight_layout()
plt.savefig("activations_plot.png", dpi=100, bbox_inches="tight")
plt.close()
print("Plot saved to activations_plot.png")

Plot saved to activations_plot.png


## Discussion: Saturation, Dead-ReLU, and Why ReLU/GELU Dominate

### Saturation (sigmoid / tanh)

Sigmoid maps all inputs to `(0, 1)` and tanh to `(-1, 1)`. Their derivatives
approach **zero** in the tails (|x| ≫ 0):

- `σ'(x) = σ(x)(1−σ(x))` → max ≈ 0.25 at x=0, → 0 as |x| → ∞.
- `tanh'(x) = 1 − tanh(x)²` → max = 1 at x=0, → 0 as |x| → ∞.

In a deep network the backpropagation gradient is multiplied by `f'(z)` at every
layer. When many units are saturated (large |z|), the gradient product shrinks
exponentially — this is the **vanishing gradient** problem
(see `[[vanishing-exploding-gradients]]`).

### Dead ReLU

ReLU's derivative is exactly **0** for all x < 0. If a neuron's pre-activation is
negative for every sample in every batch, it receives **no gradient** and never
updates. This is the *dead-ReLU* pathology, worsened by:

- Large learning rates that push weights into the permanently-negative regime.
- Bias initialised to a large negative value.

**Remedies:** LeakyReLU (α > 0 keeps a small gradient for x < 0), careful
initialisation (He / Kaiming init maintains variance for ReLU), or parameter
monitoring to detect dead units early.

### Why ReLU / GELU Dominate in Deep Nets

1. **ReLU gradient on active paths is exactly 1** — no dampening, no saturation,
   so gradient signal reaches early layers far more reliably than sigmoid/tanh.
2. **Sparsity:** roughly half of ReLU units are inactive at any forward pass,
   acting as implicit regularisation and reducing computational cost.
3. **GELU** (used in Transformers/BERT/GPT) is smooth and non-monotone:
   it attenuates small negative values softly rather than hard-clamping them,
   producing richer gradient flow and empirically better performance in
   attention-based architectures.
4. Both are computationally cheap compared to sigmoid/tanh (no exp in the
   backward pass for ReLU).

### Activation–Initialisation Pairing

| Activation | Recommended init | Reason |
|---|---|---|
| sigmoid / tanh | Xavier/Glorot | Keeps activation variance ≈ 1 at init |
| ReLU | He (Kaiming) | Accounts for half the units being zero |
| GELU / SiLU | He or slight variance scaling | Similar reasoning to ReLU |

Cross-links: `[[backpropagation]]` · `[[vanishing-exploding-gradients]]` ·
`[[batch-normalization]]` · `[[gradient-descent]]`

## Takeaways

1. **Sigmoid and tanh saturate** — derivatives vanish for large magnitudes, making
   deep-net training brittle without careful initialisation or normalisation.
2. **ReLU fixes saturation on active paths** (gradient = 1) but introduces
   **dead units** (gradient = 0 for all x < 0).
3. **LeakyReLU** prevents dead units with a small non-zero slope for x < 0.
4. **GELU** provides a smooth, probabilistic gate that outperforms ReLU in many
   modern architectures; it has non-zero gradient everywhere.
5. **Activation choice interacts with initialisation**: He init is required for
   ReLU-family activations; Xavier init for tanh/sigmoid families.
6. Every activation derivative is a *local gain factor* in backprop — poor gain
   (near 0 or ≫ 1) across many layers causes vanishing or exploding gradients.